# Miles Nordwall, Nathan Nail ML Midterm Project
TODO
- [ ] Run all best HPs for 1k and 8k iterations
- [ ] set just just .py code to work
- [ ] Description of implementation
- [ ] List best hyperparameter settings
    - [x] MNIST?
    - [ ] Fashion 8/12
    - [ ] Run 15k for all if time allows
- [ ] Task3 Tables + Explanation
- [ ] Task4 compare 8 SVCs bootstrapped vs SVC results from Task3
- [ ] Plots

## Decription of implementation

We standardized the data using sklearn's StandardScaler() which ...


In [18]:
import numpy as np
import pandas as pd
import os
import time
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import mnist, fashion_mnist

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

SEED = 96
#ITERATIONS = [1000,5000,10000]
GAMMAs = ["scale", 0.005, 0.01, 0.01, 0.001, 0.0005]
KERNALS = ['linear', 'rbf', 'poly']
Cs = [2,0.1,4,0.1,0.5,1]
PCA_DIMS = [50,100,200]
ITERATIONS = [1000, 3000, 8000]
DEGREEs = [2,2,2]


FILE_PATHS = {"mnist" :   {"pca50" : './results/mnist/pca50.csv',
                          "pca100"  : './results/mnist/pca100.csv',
                          "pca200"  : './results/mnist/pca200.csv',
                          "lda"  : './results/mnist/lda.csv',
                          "linear"  : './results/mnist/linear.csv',
                          "rbf"  : './results/mnist/rbf.csv',
                          "poly"  : './results/mnist/poly.csv'},
              "fashion" : {"pca50" : './results/fashion/pca50.csv',
                          "pca100"  : './results/fashion/pca100.csv',
                          "pca200"  : './results/fashion/pca200.csv',
                          "lda"  : './results/fashion/lda.csv',
                          "linear"  : './results/fashion/linear.csv',
                          "rbf"  : './results/fashion/rbf.csv',
                          "poly"  : './results/fashion/poly.csv'}
              }


In [20]:
'''#1'''
# MNIST - 70,000 images (60k are for training) of handwritten digits from 0-9.
(X_train, Y_train), (X_test, Y_test) = mnist.load_data()
(X_train_f, Y_train_f), (X_test_f, Y_test_f) = fashion_mnist.load_data()

In [22]:
'''#2'''
#flatten all data
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

X_train_f = X_train_f.reshape(X_train_f.shape[0], -1)
X_test_f = X_test_f.reshape(X_test_f.shape[0], -1)

In [24]:
standard_pl : Pipeline
fashion_pl : Pipeline

In [26]:
def save_n_print_results(results, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    if isinstance(results, dict):
        results = [results]  
    df = pd.concat([pd.DataFrame(r) for r in results if r is not None], ignore_index=True)
    df.to_csv(file_path, mode='a', header=not os.path.exists(file_path), index=False)
    print(df.to_string(index=False))

def plot_t_vs_acc(times, scores, num_iter):
    '''
    Params : times (1D float array) - time cost
             scores (1D float array) - accuracy 
    Makes simple plot.
    Returns : 
    '''
    xs = np.array(times)
    ys = np.array(scores)

    plt.plot(xs, ys)
    return plt
    

def fit_get_time(pipe, x_train, y_train):
    '''
    Params : pipe (Pipeline) - pipeline
             x_train (1D array) - feature training set
             y_train (1D array) - label training set

    Fits pipeline, records the time interval.

    Returns : fit pipeline (1Darray),
              total time (int)
    '''
    start = time.time()
    fit_pipe = pipe.fit(X=x_train, y=y_train)
            
    stop = time.time() 
    time_total = stop - start
    return fit_pipe, time_total

def make_pipe_(sc, dim_red, kern, num_iter, 
               c, gam = 'scale', deg = 3):
    '''
    Params : sc (StandardScaler) - standardization method
             dim_red (PCA | LDA)- dimension reduction method
             kern ('linear' | 'rbf' | 'poly') - kernal
             num_iter (int) - max iterations
             C (float) - SVC regularization hyperparameter 
             Gamma (String | float) - 
    Utility function that makes a pipeline based
    on C, Gamma, and Degree. 

    Returns : Pipeline 

    '''
    pipe = Pipeline([("scaler",sc),
                    ("dim_red",dim_red),
                    ("model", SVC(C=c, gamma=gam, degree=deg, kernel=kern, max_iter=num_iter))],
                    verbose=True)
    return pipe
            

def svc_train(dim_red, kern, sc, data, iterations,fp, shuffle=True):
    '''
    params: dim_red - type of dimension reduction
            kern - kernal 
            sc - StandardScaler
            data - data[0] : X_train
                   data[1] : Y_train
                   data[2] : X_test
                   data[3] : Y_test
            fp - file-path

    Creates the various pipelines, fits, scores, and
    stores all the results in the results_ dictionary 

    Returns : dictionary storing: 
              "dim_red" : dim_red,
              "kernal" : kern,
              "num_iter" : [],
              "cs" : [],
              "gammas" : [],
              "degs" : [],
              "pipes" : [],
              "scores" : [],
              "times" : []
    '''
    if len(Cs) == len(GAMMAs) == len(DEGREEs):
        num_hyperparams = len(Cs)
    else:
        raise ValueError(f"Cs, GAMMAs, DEGREEs must be same length. Got {len(Cs)}, {len(GAMMAs)}, {len(DEGREEs)}")

    #seperate data into train and test sets
    x_train = data[0]
    y_train = data[1]
    x_test = data[2]
    y_test = data[3]


    results_ = {
            "dim_red" : [],
            "kernal" : [],
            "num_iter" : [],
            "cs" : [],
            "gammas" : [],
            "degs" : [],
            "train_scores" : [],
            "test_scores" : [],
            "pipes" : [],
            "times" : []
            }

    print(f"Starting svc_train({str(dim_red)}{kern}{iterations})")
    #shuffle each of the global hyperparameter arrays to save time
    #instead of GridSearch
    #cs      = rand_gen.permutation(Cs)
    #gams    = rand_gen.permutation(GAMMAs)
    #degs    = rand_gen.permutation(DEGREEs)
    if shuffle:
        #no seed, want random to randomly shuffle the arrays 
        print("Permutating hyperparameters...")
        rand_gen = np.random.RandomState()
        idx_c   = rand_gen.permutation(len(Cs))
        idx_g   = rand_gen.permutation(len(GAMMAs))
        idx_d   = rand_gen.permutation(len(DEGREEs))

        cs   = [Cs[i]      for i in idx_c]
        gams = [GAMMAs[i]  for i in idx_g]
        degs = [DEGREEs[i] for i in idx_d]

    
        print(cs)
        print(gams)
        print(degs)
    else:
        cs   = Cs
        gams = GAMMAs
        degs = DEGREEs
    #cs = Cs
    #gams = GAMMAs
    #degs = DEGREEs

    
    #make pipe -> fit -> get scores -> store results
    for iteration in iterations:   
        for j in range(num_hyperparams):
            c = cs[j]
            gam = gams[j]
            deg = degs[j]
            print("---------------------------------------- Starting... ----------------------------------------")
            print(f"dim_red={str(dim_red)} | num_iter={iteration} | kernal={kern} | C={c} | gamma={gam} | degree={deg}")
            if kern == "linear":
                pipe = make_pipe_(sc=sc, dim_red=dim_red, kern=kern, num_iter=iteration,
                                  c=c)
            elif kern == "rbf":
                pipe = make_pipe_(sc=sc, dim_red=dim_red, kern=kern, num_iter=iteration,
                                  c=c, gam=gam)
            else:   #poly
                pipe = make_pipe_(sc=sc, dim_red=dim_red, kern=kern, num_iter=iteration,
                                  c=c, gam=gam, deg=deg)

            fit_pipe, time_ = fit_get_time(pipe, x_train, y_train)

            train_score =  fit_pipe.score(x_train, y_train)
            test_score =  fit_pipe.score(x_test, y_test)
            

            results_['dim_red'].append(str(dim_red))
            results_['kernal'].append(kern)
            results_['num_iter'].append(iteration)
            results_['cs'].append(c)
            results_['gammas'].append(gam)
            results_['degs'].append(deg)
            results_['pipes'].append(pipe)
            results_['train_scores'].append(train_score)
            results_['test_scores'].append(test_score)
            results_['times'].append(time_)
        
        
            print(f"dim_red={str(dim_red)} | num_iter={iteration} | kernal={kern} | C={c} | gamma={gam} | degree={deg} | time={time_} | train_score= {train_score} | test_score= {test_score}")
            print("---------------------------------------- Complete! ----------------------------------------\n\n")
        
        save_n_print_results(results_, fp)    
    
    return results_


## Best Hyperparamters
Standardization: StandardScaler()

- Made 3 global arrays : Cs, GAMMAs, and DEGREEs
- To get a feel for the hyperparameters, put a wide range of values in each
- Cs[0.0001-100.0] GAMMAs['scale', 'auto', 0.001-10.0] DEGREEs [1-10]
- For the first few rounds of testing, the 3 arrays would be shuffled for the sake of exploration.
- Took insights and 
    - Ran on linear first, found some more useful params
    - Ran on rbf next, tried

- C does
- Gamma does
- Degree does

### MNIST



### Fashion:



## PCA[50,100,200] vs LDA
- [ ] Table
    - [ ] Time cost of training
    - [ ] Dim Reduction
    - [ ] Test Error
- [ ] Explanation

## SVC Kernels : 'linear' vs 'rbf' vs 'poly'
- [ ] Table
    - [ ] Time cost of training (SVC training time only)
    - [ ] Prediciton error



In [29]:


def run_all_variations(data, folder_name, kerns, pca_comps, num_iter, fpaths):
    '''
    Params: folder_name (String) -  either "fashion" or "mnist"
            data - data[0] : X train
                   data[1] : Y train
                   data[2] : X test
                   data[3] : Y test
            kernals (1D int arr) - list of SVC kernals
            pca_dims (1D int arr) - list of n_components to pass to PCA
            num_iter (1D int arr) - list of max_iterations to run 
            fpaths (1D string arr) - list of file paths 
    '''
    if folder_name != 'fashion' and folder_name != 'mnist':
        raise ValueError(f'Expected either folder_name fashion or mnist but got: {folder_name}')
    
    all_pipes = []
    
    for k in kerns:                       # run for all kernal types
        pca_lda_results = []
        for p in pca_comps:               # run for all PCA dimensions 
            pca =  PCA(n_components=p)
            sc = StandardScaler()
            
            if p==50:
                results = svc_train(dim_red=pca,
                                        sc=sc,
                                        kern=k,
                                        data=data,
                                        iterations=num_iter,
                                        fp=fpaths[f'{folder_name}']['pca50'])
            elif p==100: 
                results = svc_train(dim_red=pca,
                                        sc=sc,
                                        kern=k,
                                        data=data,
                                        iterations=num_iter,
                                        fp=fpaths[f'{folder_name}']['pca100'])
            elif p==200: 
                results = svc_train(dim_red=pca,
                                        sc=sc,
                                        kern=k,
                                        data=data,
                                        iterations=num_iter,
                                        fp=fpaths[f'{folder_name}']['pca200'])
                
            pca_lda_results.append(results)
            all_pipes.extend(results['pipes'])
            
        # run LDA once for every 3xPCA
        sc = StandardScaler()
        lda = LinearDiscriminantAnalysis()
        lda_results = svc_train(dim_red=lda,
                                sc=sc,
                                kern=k,
                                data=data,
                                iterations=num_iter,
                                fp=fpaths[f'{folder_name}']['lda'])
        
        all_pipes.extend(lda_results['pipes'])
        pca_lda_results.append(lda_results)
        
        if k=='linear': save_n_print_results(pca_lda_results,   #save linear
                             fpaths[f'{folder_name}']['linear'])
        elif k=='rbf': save_n_print_results(pca_lda_results,    #save rbf
                             fpaths[f'{folder_name}']['rbf'])
        elif k=='poly': save_n_print_results(pca_lda_results,   #save poly
                             fpaths[f'{folder_name}']['poly'])
    
    
    return all_pipes
    

| SVC_kernal | Dim_reduction | C    | gamma | degree | train_score | test_score | iterations | time | model_time |
| ---------- | ------------- | ---- | ----- | ------ | ----------- | ---------- | ---------- | ---- | ---------- |
| 'linear'   | PCA50         | 0.01 |   |   | NEED | 0.8539 | 5000 | 99.7379 | NEED |
| 'linear'   | PCA100        | 
| 'linear'   | PCA200        |
| 'linear'   | LDA           | 0.01 |      |      | NEED | 0.8255 | 5000 |19.5266 | NEED | 
| 'rbf'      | PCA50         | 2.0 | 0.005 |    | 0.93725 | 0.8802 | 3000 | 54.4010 |  47.1s |
| 'rbf'      | PCA100        | 2.0 | 0.005 |    | 0.959433 | 0.8854 | 3000 | 112.2138 | 1.7min | 
| 'rbf'      | PCA200        | 2.0 | 0.005 |    | 0.97615 | 0.8863 | 3000 | 248.997985 | 3.9min | 
| 'rbf'      | LDA           | 1.5 | 0.005 |    |  0.847  | 0.8286 | 3000 | 29.92428 | 10.7s | 
| 'poly'     | PCA50         | 2.0 | 0.005 | 2 |  0.868633  | 0.8712 | 3000 | 37.753571 | 32.1s | 
| 'poly'     | PCA100        | 2.0 | scale | 3 |  0.907400  | 0.8511 | 3000 | 59.641161 | 54.6s | 
| 'poly'     | PCA100        | 4.0 | 0.001 | 3 |  0.896283   | 0.8645 | 3000 | 65.252791 | 1.0min |
| 'poly'     | PCA200        | 2.0 | scale | 4 |  0.896967   | 0.8450 | 3000 | 177.180348 | 2.8min | 
| 'poly'     | PCA200        | 10.0 | 0.001 | 2 |  0.927783    | 0.8799 | 3000 | 101.437330 | 1.6min | 
| 'poly'     | LDA           | 2.0 | scale | 4 |  0.847300    | 0.8264 | 3000 | 20.346974 | 7.8s | 
| 'poly'     | LDA           | 10.0 | 0.005 | 2 |  0.841967    | 0.8238 | 3000 | 20.690787 | 8.2s | 

| SVC_kernal | Dim_reduction | C   | gamma | degree | train_score | test_score | iterations | time     |
| ---------- | ------------- | --- | ----- | ------ | ----------- | ---------- | ---------- | -------- |
| 'linear'   | PCA50         | 0.05 |   |        | 0.7711      | 0.7553     | 3000       | 33.28s  |
| 'linear'   | PCA100        | 0.05 |   |        | 0.7966      | 0.7771     | 3000       | 56.08s  |
| 'linear'   | PCA200        | 0.05 |   |       | 0.7853      | 0.7665     | 3000       | 103.02s |
| 'linear'   | LDA           | 0.05 |   |       | 0.8436      | 0.8257     | 3000       | 27.99s  |
| 'rbf'      | PCA50         | 2.0 | 0.005 |    | 0.93725 | 0.8802 | 3000 | 54.4010 |  
| 'rbf'      | PCA100        | 2.0 | 0.005 |    | 0.959433 | 0.8854 | 3000 | 112.2138 | 
| 'rbf'      | PCA200        | 2.0 | 0.005 |       | 0.9762      | 0.8863     | 3000       | 249.00s  |
| 'rbf'      | LDA           | 2.0 | 0.005 |       | 0.8475      | 0.8285     | 3000       | 33.20s   |
| 'poly'     | PCA50         | 5   | 0.0005 | 2      | 0.8686      | 0.8511     | 3000       | 37.75s  |
| 'poly'     | PCA100        | 2   | scale  | 3      | 0.9074      | 0.8712     | 3000       | 59.64s  |
| 'poly'     | PCA200        | 10  | 0.001  | 2      | 0.9278      | 0.8799     | 3000       | 101.44s |
| 'poly'     | LDA           | 5   | 0.005  | 2      | 0.8396      | 0.8216     | 3000       | 21.71s  |

| SVC_kernal | Dim_reduction | C    | gamma  | degree | train_score | test_score | iterations | time    |
| ---------- | ------------- | ---- | ------ | ------ | ----------- | ---------- | ---------- | ------- |
| 'linear'   | PCA50         | 0.05 | 0.005  | 2      | 0.7711      | 0.7553     | 3000       | 33.28s  |
| 'linear'   | PCA100        | 0.05 | 0.0001 | 2      | 0.7966      | 0.7771     | 3000       | 56.08s  |
| 'linear'   | PCA200        | 0.05 | 0.005  | 2      | 0.7853      | 0.7665     | 3000       | 103.02s |
| 'linear'   | LDA           | 0.05 | 0.005  | 2      | 0.8436      | 0.8257     | 3000       | 27.99s  |


                     dim_red kernal  num_iter  cs  gammas  degs  train_scores  test_scores                                                                                    pipes      times
        PCA(n_components=50)    rbf      1000 2.0   0.005     2      0.909650       0.8592         (StandardScaler(), PCA(n_components=50), SVC(C=2.0, gamma=0.005, max_iter=1000))  42.202408
        PCA(n_components=50)    rbf      3000 2.0   0.005     2      0.937317       0.8799         (StandardScaler(), PCA(n_components=50), SVC(C=2.0, gamma=0.005, max_iter=3000))  51.319073
        PCA(n_components=50)    rbf      8000 2.0   0.005     2      0.937167       0.8796         (StandardScaler(), PCA(n_components=50), SVC(C=2.0, gamma=0.005, max_iter=8000))  57.661265
       PCA(n_components=100)    rbf      1000 2.0   0.005     2      0.945750       0.8788        (StandardScaler(), PCA(n_components=100), SVC(C=2.0, gamma=0.005, max_iter=1000))  91.367591
       PCA(n_components=100)    rbf      3000 2.0   0.005     2      0.959667       0.8863        (StandardScaler(), PCA(n_components=100), SVC(C=2.0, gamma=0.005, max_iter=3000)) 109.765069
       PCA(n_components=100)    rbf      8000 2.0   0.005     2      0.959533       0.8860        (StandardScaler(), PCA(n_components=100), SVC(C=2.0, gamma=0.005, max_iter=8000)) 122.003563
       PCA(n_components=200)    rbf      1000 2.0   0.005     2      0.966683       0.8798        (StandardScaler(), PCA(n_components=200), SVC(C=2.0, gamma=0.005, max_iter=1000)) 176.283724
       PCA(n_components=200)    rbf      3000 2.0   0.005     2      0.976083       0.8865        (StandardScaler(), PCA(n_components=200), SVC(C=2.0, gamma=0.005, max_iter=3000)) 237.045199
       PCA(n_components=200)    rbf      8000 2.0   0.005     2      0.976700       0.8872        (StandardScaler(), PCA(n_components=200), SVC(C=2.0, gamma=0.005, max_iter=8000)) 283.129974
LinearDiscriminantAnalysis()    rbf      1000 2.0   0.005     2      0.663100       0.6528 (StandardScaler(), LinearDiscriminantAnalysis(), SVC(C=2.0, gamma=0.005, max_iter=1000))  25.488941
LinearDiscriminantAnalysis()    rbf      3000 2.0   0.005     2      0.847483       0.8285 (StandardScaler(), LinearDiscriminantAnalysis(), SVC(C=2.0, gamma=0.005, max_iter=3000))  29.571348
LinearDiscriminantAnalysis()    rbf      8000 2.0   0.005     2      0.847500       0.8285 (StandardScaler(), LinearDiscriminantAnalysis(), SVC(C=2.0, gamma=0.005, max_iter=8000))  28.099993

In [ ]:
'''fashion'''
#GAMMAs = [0.0005,0.005,0.005,0.001]
#DEGREEs = [1,1,2,2,2,3,4,4,5]
Cs = [2,5,10]
GAMMAs = [0.001,0.0005,0.0001]
KERNALS = ['poly']
#Cs = [2,2,4,5,10,20]
f_pipes = run_all_variations([X_train_f, Y_train_f, X_test_f, Y_test_f], 
                             folder_name='fashion', 
                             kerns=KERNALS, 
                             pca_comps=PCA_DIMS, 
                             num_iter=ITERATIONS, 
                             fpaths=FILE_PATHS)


Starting svc_train(PCA(n_components=50)poly[1000, 3000, 8000])
Permutating hyperparameters...
[10, 5, 2]
[0.0005, 0.001, 0.0001]
[2, 2, 2]
---------------------------------------- Starting... ----------------------------------------
dim_red=PCA(n_components=50) | num_iter=1000 | kernal=poly | C=10 | gamma=0.0005 | degree=2
[Pipeline] ............ (step 1 of 3) Processing scaler, total=   0.5s
[Pipeline] ........... (step 2 of 3) Processing dim_red, total=   4.2s


/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


[Pipeline] ............. (step 3 of 3) Processing model, total=  25.3s
dim_red=PCA(n_components=50) | num_iter=1000 | kernal=poly | C=10 | gamma=0.0005 | degree=2 | time=29.95417809486389 | train_score= 0.6245 | test_score= 0.6176
---------------------------------------- Complete! ----------------------------------------


---------------------------------------- Starting... ----------------------------------------
dim_red=PCA(n_components=50) | num_iter=1000 | kernal=poly | C=5 | gamma=0.001 | degree=2
[Pipeline] ............ (step 1 of 3) Processing scaler, total=   0.4s
[Pipeline] ........... (step 2 of 3) Processing dim_red, total=   4.1s


/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


[Pipeline] ............. (step 3 of 3) Processing model, total=  22.8s
dim_red=PCA(n_components=50) | num_iter=1000 | kernal=poly | C=5 | gamma=0.001 | degree=2 | time=27.293976068496704 | train_score= 0.6707666666666666 | test_score= 0.6622
---------------------------------------- Complete! ----------------------------------------


---------------------------------------- Starting... ----------------------------------------
dim_red=PCA(n_components=50) | num_iter=1000 | kernal=poly | C=2 | gamma=0.0001 | degree=2
[Pipeline] ............ (step 1 of 3) Processing scaler, total=   0.4s
[Pipeline] ........... (step 2 of 3) Processing dim_red, total=   3.9s


/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


[Pipeline] ............. (step 3 of 3) Processing model, total=  58.0s
dim_red=PCA(n_components=50) | num_iter=1000 | kernal=poly | C=2 | gamma=0.0001 | degree=2 | time=62.39735293388367 | train_score= 0.29401666666666665 | test_score= 0.2962
---------------------------------------- Complete! ----------------------------------------


             dim_red kernal  num_iter  cs  gammas  degs  train_scores  test_scores                                                                                                     pipes     times
PCA(n_components=50)   poly      1000  10  0.0005     2      0.624500       0.6176 (StandardScaler(), PCA(n_components=50), SVC(C=10, degree=2, gamma=0.0005, kernel='poly', max_iter=1000)) 29.954178
PCA(n_components=50)   poly      1000   5  0.0010     2      0.670767       0.6622   (StandardScaler(), PCA(n_components=50), SVC(C=5, degree=2, gamma=0.001, kernel='poly', max_iter=1000)) 27.293976
PCA(n_components=50)   poly      1000   2  0.0001     2      0.294

/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=3000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


[Pipeline] ............. (step 3 of 3) Processing model, total=  35.0s


In [11]:
'''fashion'''
#GAMMAs = [0.0005,0.005,0.005,0.001]
#DEGREEs = [1,1,2,2,2,3,4,4,5]
Cs = [ 2.0]
GAMMAs = [0.005]
KERNALS = ['rbf']
#Cs = [2,2,4,5,10,20]
f_pipes = run_all_variations([X_train_f, Y_train_f, X_test_f, Y_test_f], 
                             folder_name='fashion', 
                             kerns=KERNALS, 
                             pca_comps=PCA_DIMS, 
                             num_iter=ITERATIONS, 
                             fpaths=FILE_PATHS)


Starting svc_train(PCA(n_components=50)rbf[1000, 3000, 8000])
Permutating hyperparameters...
[2.0]
[0.005]
[2]
---------------------------------------- Starting... ----------------------------------------
dim_red=PCA(n_components=50) | num_iter=1000 | kernal=rbf | C=2.0 | gamma=0.005 | degree=2
[Pipeline] ............ (step 1 of 3) Processing scaler, total=   0.3s
[Pipeline] ........... (step 2 of 3) Processing dim_red, total=   8.7s


/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


[Pipeline] ............. (step 3 of 3) Processing model, total=  33.1s
dim_red=PCA(n_components=50) | num_iter=1000 | kernal=rbf | C=2.0 | gamma=0.005 | degree=2 | time=42.20240783691406 | train_score= 0.90965 | test_score= 0.8592
---------------------------------------- Complete! ----------------------------------------


             dim_red kernal  num_iter  cs  gammas  degs  train_scores  test_scores                                                                            pipes     times
PCA(n_components=50)    rbf      1000 2.0   0.005     2       0.90965       0.8592 (StandardScaler(), PCA(n_components=50), SVC(C=2.0, gamma=0.005, max_iter=1000)) 42.202408
---------------------------------------- Starting... ----------------------------------------
dim_red=PCA(n_components=50) | num_iter=3000 | kernal=rbf | C=2.0 | gamma=0.005 | degree=2
[Pipeline] ............ (step 1 of 3) Processing scaler, total=   0.3s
[Pipeline] ........... (step 2 of 3) Processing dim_red, total=   9.8s


/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=3000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


[Pipeline] ............. (step 3 of 3) Processing model, total=  41.2s
dim_red=PCA(n_components=50) | num_iter=3000 | kernal=rbf | C=2.0 | gamma=0.005 | degree=2 | time=51.31907296180725 | train_score= 0.9373166666666667 | test_score= 0.8799
---------------------------------------- Complete! ----------------------------------------


             dim_red kernal  num_iter  cs  gammas  degs  train_scores  test_scores                                                                            pipes     times
PCA(n_components=50)    rbf      1000 2.0   0.005     2      0.909650       0.8592 (StandardScaler(), PCA(n_components=50), SVC(C=2.0, gamma=0.005, max_iter=1000)) 42.202408
PCA(n_components=50)    rbf      3000 2.0   0.005     2      0.937317       0.8799 (StandardScaler(), PCA(n_components=50), SVC(C=2.0, gamma=0.005, max_iter=3000)) 51.319073
---------------------------------------- Starting... ----------------------------------------
dim_red=PCA(n_components=50) | num_iter=8000 | k

...

[Pipeline] ............. (step 3 of 3) Processing model, total= 4.5min
dim_red=PCA(n_components=200) | num_iter=8000 | kernal=rbf | C=2.0 | gamma=0.005 | degree=2 | time=283.1299741268158 | train_score= 0.9767 | test_score= 0.8872
---------------------------------------- Complete! ----------------------------------------


              dim_red kernal  num_iter  cs  gammas  degs  train_scores  test_scores                                                                             pipes      times
PCA(n_components=200)    rbf      1000 2.0   0.005     2      0.966683       0.8798 (StandardScaler(), PCA(n_components=200), SVC(C=2.0, gamma=0.005, max_iter=1000)) 176.283724
PCA(n_components=200)    rbf      3000 2.0   0.005     2      0.976083       0.8865 (StandardScaler(), PCA(n_components=200), SVC(C=2.0, gamma=0.005, max_iter=3000)) 237.045199
PCA(n_components=200)    rbf      8000 2.0   0.005     2      0.976700       0.8872 (StandardScaler(), PCA(n_components=200), SVC(C=2.0, gamma=0

/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


[Pipeline] ............. (step 3 of 3) Processing model, total=   7.4s
dim_red=LinearDiscriminantAnalysis() | num_iter=1000 | kernal=rbf | C=2.0 | gamma=0.005 | degree=2 | time=25.488940954208374 | train_score= 0.6631 | test_score= 0.6528
---------------------------------------- Complete! ----------------------------------------


                     dim_red kernal  num_iter  cs  gammas  degs  train_scores  test_scores                                                                                    pipes     times
LinearDiscriminantAnalysis()    rbf      1000 2.0   0.005     2        0.6631       0.6528 (StandardScaler(), LinearDiscriminantAnalysis(), SVC(C=2.0, gamma=0.005, max_iter=1000)) 25.488941
---------------------------------------- Starting... ----------------------------------------
dim_red=LinearDiscriminantAnalysis() | num_iter=3000 | kernal=rbf | C=2.0 | gamma=0.005 | degree=2
[Pipeline] ............ (step 1 of 3) Processing scaler, total=   0.4s
[Pipeline] ........... 

/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=3000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


[Pipeline] ............. (step 3 of 3) Processing model, total=  10.7s
dim_red=LinearDiscriminantAnalysis() | num_iter=3000 | kernal=rbf | C=2.0 | gamma=0.005 | degree=2 | time=29.571347951889038 | train_score= 0.8474833333333334 | test_score= 0.8285
---------------------------------------- Complete! ----------------------------------------


                     dim_red kernal  num_iter  cs  gammas  degs  train_scores  test_scores                                                                                    pipes     times
LinearDiscriminantAnalysis()    rbf      1000 2.0   0.005     2      0.663100       0.6528 (StandardScaler(), LinearDiscriminantAnalysis(), SVC(C=2.0, gamma=0.005, max_iter=1000)) 25.488941
LinearDiscriminantAnalysis()    rbf      3000 2.0   0.005     2      0.847483       0.8285 (StandardScaler(), LinearDiscriminantAnalysis(), SVC(C=2.0, gamma=0.005, max_iter=3000)) 29.571348
---------------------------------------- Starting... --------------------------------

| SVC_kernal | Dim_reduction | C   | gamma | degree | train_score | test_score | iterations | time     |
| ---------- | ------------- | --- | ----- | ------ | ----------- | ---------- | ---------- | -------- |
| 'linear'   | PCA50         | 0.1 |  |       | 0.8903      | 0.8920     | 3000       | 29.88s   |
| 'linear'   | PCA100        | 0.1 |  |       | 0.8999      | 0.8952     | 3000       | 53.28s   |
| 'linear'   | PCA200        | 0.1 |  |       | 0.9010      | 0.8944     | 3000       | 96.32s   |
| 'linear'   | LDA           | 0.1 |   |       | 0.8974      | 0.8929     | 3000       | 30.75s   |
| 'rbf'      | PCA50         | 2.0 | 0.005 |       | 0.9943      | 0.9728     | 3000       | 45.33s   |
| 'rbf'      | PCA100        | 4.0 | scale |      | 0.9938      | 0.9777     | 3000       | 54.32s   |
| 'rbf'      | PCA200        | 4.0 | 0.001 |       | 0.9898      | 0.9744     | 3000       | 91.54s   |
| 'rbf'      | LDA           | 4.0 | 0.005 |       | 0.9120      | 0.9054     | 3000       | 17.05s   |
| 'poly'     | PCA50         | 1.0 | 0.005 | 2      | 0.9861      | 0.9728     | 3000       | 20.07s   |
| 'poly'     | PCA100        | 0.1 | 0.01  | 3      | 0.9961      | 0.9773     | 3000       | 45.16s   |
| 'poly'     | PCA200        | 0.1 | 0.01  | 2      | 0.9926      | 0.9775     | 3000       | 88.37s   |
| 'poly'     | LDA           | 0.1 | scale | 2      | 0.8773      | 0.8697     | 3000       | 22.68s   |


In [13]:
'''mnist'''
m_pipes = run_all_variations([X_train, Y_train, X_test, Y_test], 
                             folder_name='mnist', 
                             kerns=KERNALS, 
                             pca_comps=PCA_DIMS, 
                             num_iter=ITERATIONS, 
                             fpaths=FILE_PATHS)


Starting svc_train(PCA(n_components=50)rbf[1000, 3000, 8000])
Permutating hyperparameters...
[2.0]
[0.005]
[2]
---------------------------------------- Starting... ----------------------------------------
dim_red=PCA(n_components=50) | num_iter=1000 | kernal=rbf | C=2.0 | gamma=0.005 | degree=2
[Pipeline] ............ (step 1 of 3) Processing scaler, total=   0.5s
[Pipeline] ........... (step 2 of 3) Processing dim_red, total=   8.6s


/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


[Pipeline] ............. (step 3 of 3) Processing model, total=  36.3s
dim_red=PCA(n_components=50) | num_iter=1000 | kernal=rbf | C=2.0 | gamma=0.005 | degree=2 | time=45.36636424064636 | train_score= 0.9941333333333333 | test_score= 0.9734
---------------------------------------- Complete! ----------------------------------------


             dim_red kernal  num_iter  cs  gammas  degs  train_scores  test_scores                                                                            pipes     times
PCA(n_components=50)    rbf      1000 2.0   0.005     2      0.994133       0.9734 (StandardScaler(), PCA(n_components=50), SVC(C=2.0, gamma=0.005, max_iter=1000)) 45.366364
---------------------------------------- Starting... ----------------------------------------
dim_red=PCA(n_components=50) | num_iter=3000 | kernal=rbf | C=2.0 | gamma=0.005 | degree=2
[Pipeline] ............ (step 1 of 3) Processing scaler, total=   0.4s
[Pipeline] ........... (step 2 of 3) Processing dim_red, tot

/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=3000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


[Pipeline] ............. (step 3 of 3) Processing model, total=  40.7s
dim_red=PCA(n_components=50) | num_iter=3000 | kernal=rbf | C=2.0 | gamma=0.005 | degree=2 | time=47.69372582435608 | train_score= 0.9942 | test_score= 0.9727
---------------------------------------- Complete! ----------------------------------------


             dim_red kernal  num_iter  cs  gammas  degs  train_scores  test_scores                                                                            pipes     times
PCA(n_components=50)    rbf      1000 2.0   0.005     2      0.994133       0.9734 (StandardScaler(), PCA(n_components=50), SVC(C=2.0, gamma=0.005, max_iter=1000)) 45.366364
PCA(n_components=50)    rbf      3000 2.0   0.005     2      0.994200       0.9727 (StandardScaler(), PCA(n_components=50), SVC(C=2.0, gamma=0.005, max_iter=3000)) 47.693726
---------------------------------------- Starting... ----------------------------------------
dim_red=PCA(n_components=50) | num_iter=8000 | kernal=rbf | 

...

/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


[Pipeline] ............. (step 3 of 3) Processing model, total= 3.2min
dim_red=PCA(n_components=200) | num_iter=1000 | kernal=rbf | C=2.0 | gamma=0.005 | degree=2 | time=205.17707586288452 | train_score= 0.9985833333333334 | test_score= 0.9635
---------------------------------------- Complete! ----------------------------------------


              dim_red kernal  num_iter  cs  gammas  degs  train_scores  test_scores                                                                             pipes      times
PCA(n_components=200)    rbf      1000 2.0   0.005     2      0.998583       0.9635 (StandardScaler(), PCA(n_components=200), SVC(C=2.0, gamma=0.005, max_iter=1000)) 205.177076
---------------------------------------- Starting... ----------------------------------------
dim_red=PCA(n_components=200) | num_iter=3000 | kernal=rbf | C=2.0 | gamma=0.005 | degree=2
[Pipeline] ............ (step 1 of 3) Processing scaler, total=   0.5s
[Pipeline] ........... (step 2 of 3) Processing dim

/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=3000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


[Pipeline] ............. (step 3 of 3) Processing model, total= 3.8min


KeyboardInterrupt: 

In [ ]:
#Task 4 : 8(x3 kernels) SVC Voters vs 1(x3 kernels) SVC
from sklearn.model_selection import KFold

num_sets = 8

# TODO divide MNIST into 8 disjoint sets using Kfold
mnist_X_all = np.concatenate([X_train, X_test], axis=0)
mnist_Y_all = np.concatenate([Y_train, Y_test], axis=0)
kf = KFold(n_splits=num_sets, shuffle=True, random_state=SEED)

mnist_batches = {"x_trains" : [],
                 "y_trains" : [],
                 "x_tests" : [],
                 "y_tests" : []
                }
for train_index, test_index in kf.split(mnist_X_all):
    mnist_batches["x_trains"].append(mnist_X_all[train_index])
    mnist_batches["y_trains"].append(mnist_Y_all[train_index])
    mnist_batches["x_tests"].append(mnist_X_all[test_index])
    mnist_batches["y_tests"].append(mnist_Y_all[test_index])
    


fashion_X_all = np.concatenate([X_train_f, X_test_f], axis=0)
fashion_Y_all = np.concatenate([Y_train_f, Y_test_f], axis=0)
kf_f = KFold(n_splits=num_sets, shuffle=True, random_state=SEED)

fashion_batches = {"x_trains" : [],
                   "y_trains" : [],
                   "x_tests" : [],
                   "y_tests" : []
                  }


for f_train_index, f_test_index in kf_f.split(fashion_X_all):
    fashion_batches["x_trains"].append(fashion_X_all[f_train_index])
    fashion_batches["y_trains"].append(fashion_Y_all[f_train_index])
    fashion_batches["x_tests"].append(fashion_X_all[f_test_index])
    fashion_batches["y_tests"].append(fashion_Y_all[f_test_index])
    
    
# TODO final prediction result obtained by voting



In [ ]:
'''mnist bootstrap'''
m_boot_pipes = []
best_dim_red = []

for i in range(num_sets):
    data = [mnist_batches["x_trains"][i],
            mnist_batches["y_trains"][i],
            mnist_batches["x_tests"][i],
            mnist_batches["y_tests"][i],
           ]
    def svc_train(dim_red, kern, sc, data, iterations,fp, shuffle=True):
    m_pipes_ = svc_train(dim_red)
    m_pipes_ = run_all_variations(data, 
                       folder_name='mnist', 
                       kerns=KERNALS, 
                       pca_comps=best_dim_red, 
                       num_iter=ITERATIONS, 
                       fpaths=FILE_PATHS
                       shuffle-False)
                        
    m_boot_pipes.extend(m_pipes_)

In [ ]:
'''fashion bootstrap'''
f_boot_pipes = []
for i in range(num_sets):
    data = [fashion_batches["x_trains"][i],
            fashion_batches["y_trains"][i],
            fashion_batches["x_tests"][i],
            fashion_batches["y_tests"][i],
           ]
    f_pipes_ = run_all_variations(data, 
                       folder_name='fashion', 
                       kerns=KERNALS, 
                       pca_comps=PCA_DIMS, 
                       num_iter=ITERATIONS, 
                       fpaths=FILE_PATHS)
    f_boot_pipes.extend(f_pipes_)

In [ ]:
'''mnist'''
#TODO note the scale param

'''linear'''
# PCA(n_components=50) linear      3000 0.1   scale     3      0.890317       0.8920 (StandardScaler(), PCA(n_components=50), SVC(C=0.1, kernel='linear', max_iter=3000)) 29.876782
# PCA(n_components=100) linear      3000 0.1   scale     4      0.899967       0.8952 (StandardScaler(), PCA(n_components=100), SVC(C=0.1, kernel='linear', max_iter=3000)) 53.278366
# PCA(n_components=200) linear      3000 0.1   scale     4      0.900950       0.8944        (StandardScaler(), PCA(n_components=200), SVC(C=0.1, kernel='linear', max_iter=3000))  96.315316
# LinearDiscriminantAnalysis() linear      3000 0.1    0.01     2      0.897350       0.8929 (StandardScaler(), LinearDiscriminantAnalysis(), SVC(C=0.1, kernel='linear', max_iter=3000)) 30.754208

# PCA(n_components=50)    rbf      3000 2.0   0.005     2      0.994317       0.9728    (StandardScaler(), PCA(n_components=50), SVC(C=2, gamma=0.005, max_iter=3000))  45.333538
    # PCA(n_components=50)    rbf      3000 1.0   scale     2      0.981933       0.9715                 (StandardScaler(), PCA(n_components=50), SVC(C=1, max_iter=3000))  34.436837    NOTE SCALE!

'''rbf'''
                     dim_red kernal  num_iter  cs  gammas  degs  train_scores  test_scores                                                                                     pipes      times
# PCA(n_components=100)    rbf      3000 1.0   0.001     4      0.972867       0.9652           (StandardScaler(), PCA(n_components=100), SVC(C=1, gamma=0.001, max_iter=3000))  59.453224
    # PCA(n_components=100)    rbf      3000 4.0   scale     3      0.993750       0.9777                        (StandardScaler(), PCA(n_components=100), SVC(C=4, max_iter=3000))  54.315469
# PCA(n_components=200)    rbf      3000 4.0   0.001     4      0.989750       0.9744           (StandardScaler(), PCA(n_components=200), SVC(C=4, gamma=0.001, max_iter=3000))  91.539076
    # PCA(n_components=200)    rbf      3000 0.1   scale     2      0.946433       0.9414                      (StandardScaler(), PCA(n_components=200), SVC(C=0.1, max_iter=3000)) 235.223733
# LinearDiscriminantAnalysis()    rbf      3000 4.0   0.005     2      0.911967       0.9054    (StandardScaler(), LinearDiscriminantAnalysis(), SVC(C=4, gamma=0.005, max_iter=3000))  17.047276
    # LinearDiscriminantAnalysis()    rbf      3000 2.0   scale     4      0.929950       0.9230                 (StandardScaler(), LinearDiscriminantAnalysis(), SVC(C=2, max_iter=3000))  16.132000

'''poly : default degree=3''' 
                     dim_red kernal  num_iter  cs  gammas  degs  train_scores  test_scores                                                                                                            pipes      times
        
#PCA(n_components=50)   poly      3000 1.0   0.005     2      0.986067       0.9728          (StandardScaler(), PCA(n_components=50), SVC(C=1, degree=2, gamma=0.005, kernel='poly', max_iter=3000))  20.065056        
    #PCA(n_components=50)   poly      3000 4.0   scale     2      0.986283       0.9722                       (StandardScaler(), PCA(n_components=50), SVC(C=4, degree=2, kernel='poly', max_iter=3000))  20.188934
# PCA(n_components=100)   poly      3000 0.1    0.01     3      0.996100       0.9773                  (StandardScaler(), PCA(n_components=100), SVC(C=0.1, gamma=0.01, kernel='poly', max_iter=3000))  45.158754
    # PCA(n_components=100)   poly      3000 1.0   scale     3      0.979050       0.9669                                (StandardScaler(), PCA(n_components=100), SVC(C=1, kernel='poly', max_iter=3000))  87.316304
      #best NOT deg=3  # PCA(n_components=100)   poly      3000 4.0    0.01     2      0.999217       0.9720          (StandardScaler(), PCA(n_components=100), SVC(C=4, degree=2, gamma=0.01, kernel='poly', max_iter=3000))  29.576657
       
# PCA(n_components=200)   poly      3000 0.1    0.01     2      0.992600       0.9775        (StandardScaler(), PCA(n_components=200), SVC(C=0.1, degree=2, gamma=0.01, kernel='poly', max_iter=3000))  88.371280
    # PCA(n_components=200)   poly      3000 2.0   scale     2      0.989100       0.9759                      (StandardScaler(), PCA(n_components=200), SVC(C=2, degree=2, kernel='poly', max_iter=3000)) 104.344510
# LinearDiscriminantAnalysis()   poly      3000 0.5    0.01     2      0.866383       0.8628 (StandardScaler(), LinearDiscriminantAnalysis(), SVC(C=0.5, degree=2, gamma=0.01, kernel='poly', max_iter=3000))  26.266702
    # LinearDiscriminantAnalysis()   poly      3000 0.1   scale     2      0.877267       0.8697             (StandardScaler(), LinearDiscriminantAnalysis(), SVC(C=0.1, degree=2, kernel='poly', max_iter=3000))  22.677451